# Monthly data pipeline (single notebook)
This notebook downloads/builds all required monthly features and writes one final panel CSV.


## 1) Setup


In [ ]:
import os
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from pathlib import Path
from io import StringIO

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)



## 2) Futures prices and returns (corn, soybean, wheat)


In [ ]:
tickers = {
    "corn": "ZC=F",
    "soybean": "ZS=F",
    "wheat": "ZW=F",
}

frames = []
for commodity, ticker in tickers.items():
    daily = yf.download(
        ticker,
        start="2010-01-01",
        end="2026-01-01",
        auto_adjust=False,
        progress=False,
    )

    if daily.empty:
        print(f"No data returned for {commodity} ({ticker}).")
        continue

    if isinstance(daily.columns, pd.MultiIndex):
        daily.columns = daily.columns.get_level_values(0)

    price_col = "Adj Close" if "Adj Close" in daily.columns else "Close"
    if price_col not in daily.columns:
        print(f"No price column available for {commodity} ({ticker}).")
        continue

    monthly = daily[[price_col]].resample("ME").last().copy()
    monthly["futures_return"] = np.log(monthly[price_col]).diff()
    monthly["commodity"] = commodity
    monthly = monthly.reset_index().rename(columns={"Date": "date", price_col: "futures_price"})
    frames.append(monthly)

if not frames:
    raise RuntimeError("No commodity data downloaded. Check internet connection and ticker symbols.")

futures_df = pd.concat(frames, ignore_index=True)
futures_df.to_csv("data/raw/futures_monthly.csv", index=False)
print("saved data/raw/futures_monthly.csv", futures_df.shape)


## 3) Macro variables from FRED (USD index, interest rate, VIX)


In [ ]:
# FRED API uses Bearer token header.
# Before running, set your key in terminal:
# export FRED_API_KEY="<your_key>"
fred_api_key = os.getenv("FRED_API_KEY")
if not fred_api_key:
    raise RuntimeError("Set FRED_API_KEY before running this notebook.")

fred_url = "https://api.stlouisfed.org/fred/series/observations"
series_map = {
    "usd_index": "DTWEXBGS",
    "interest_rate": "TB3MS",
    "vix": "VIXCLS",
}

macro_parts = []
for output_name, series_id in series_map.items():
    params = {
        "series_id": series_id,
        "file_type": "json",
        "observation_start": "2010-01-01",
    }
    headers = {
        "Authorization": f"Bearer {fred_api_key}",
    }

    r = requests.get(fred_url, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    obs = r.json().get("observations", [])

    df = pd.DataFrame(obs)
    if df.empty:
        df = pd.DataFrame(columns=["date", output_name])
    else:
        df = df[["date", "value"]].copy()
        df["date"] = pd.to_datetime(df["date"])
        df[output_name] = pd.to_numeric(df["value"], errors="coerce")
        df = df[["date", output_name]]

    # daily -> monthly mean for daily series
    if output_name in ["usd_index", "vix"]:
        df = df.set_index("date").resample("ME").mean().reset_index()
    else:
        df["date"] = pd.to_datetime(df["date"]).dt.to_period("M").dt.to_timestamp("M")

    macro_parts.append(df)

macro_df = macro_parts[0]
macro_df = macro_df.merge(macro_parts[1], on="date", how="outer")
macro_df = macro_df.merge(macro_parts[2], on="date", how="outer")
macro_df = macro_df.sort_values("date").reset_index(drop=True)
macro_df.to_csv("data/raw/macro_monthly.csv", index=False)
print("saved data/raw/macro_monthly.csv", macro_df.shape)




## 4) ENSO + climate/disaster inputs


In [ ]:
# Set this to True only when your network can reach NOAA CPC reliably.
USE_ONLINE_ONI = False
oni_url = "https://origin.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/ONI_v5.php"
season_to_month = {
    "DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4,
    "AMJ": 5, "MJJ": 6, "JJA": 7, "JAS": 8,
    "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12,
}

climate_df = pd.DataFrame(columns=["date", "enso_oni"])

if USE_ONLINE_ONI:
    oni_table = None
    last_error = None
    for attempt in range(3):
        try:
            resp = requests.get(oni_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
            resp.raise_for_status()
            oni_table = pd.read_html(StringIO(resp.text))[0].copy()
            break
        except Exception as e:
            last_error = e
            print(f"ONI download attempt {attempt + 1}/3 failed: {e}")

    if oni_table is not None:
        if "Year" not in oni_table.columns:
            oni_table.columns = [str(c).strip() for c in oni_table.columns]
        oni_long = oni_table.melt(id_vars=["Year"], var_name="season", value_name="enso_oni")
        oni_long["month"] = oni_long["season"].map(season_to_month)
        oni_long = oni_long.dropna(subset=["month"])
        oni_long["date"] = pd.to_datetime(
            dict(year=oni_long["Year"].astype(int), month=oni_long["month"].astype(int), day=1)
        ) + pd.offsets.MonthEnd(0)
        oni_long["enso_oni"] = pd.to_numeric(oni_long["enso_oni"], errors="coerce")
        climate_df = oni_long[["date", "enso_oni"]].copy()
    else:
        print(f"Online ONI unavailable after retries: {last_error}")

# Local ONI fallback (recommended for unstable network)
local_oni = Path("data/raw/oni_monthly_input.csv")
if climate_df.empty and local_oni.exists():
    oni_local_df = pd.read_csv(local_oni)
    if "date" not in oni_local_df.columns or "enso_oni" not in oni_local_df.columns:
        raise ValueError("data/raw/oni_monthly_input.csv must have columns: date, enso_oni")
    climate_df = oni_local_df[["date", "enso_oni"]].copy()
    climate_df["date"] = pd.to_datetime(climate_df["date"]).dt.to_period("M").dt.to_timestamp("M")
    print("Used local ONI: data/raw/oni_monthly_input.csv")

if climate_df.empty:
    print("No ONI source available. Continuing with empty enso_oni column.")

# optional monthly inputs (if files exist)
optional_files = [
    ("data/raw/drought_monthly_input.csv", "drought_index"),
    ("data/raw/temp_anomaly_input.csv", "temperature_anomaly"),
    ("data/raw/precip_anomaly_input.csv", "precipitation_anomaly"),
    ("data/raw/extreme_heat_input.csv", "extreme_heat_events"),
]

for file_path, col_name in optional_files:
    if Path(file_path).exists():
        temp_df = pd.read_csv(file_path)
        if "date" not in temp_df.columns or col_name not in temp_df.columns:
            raise ValueError(f"{file_path} must include columns date and {col_name}")
        temp_df = temp_df[["date", col_name]].copy()
        temp_df["date"] = pd.to_datetime(temp_df["date"]).dt.to_period("M").dt.to_timestamp("M")
        climate_df = climate_df.merge(temp_df, on="date", how="outer")
    else:
        print(f"missing optional file: {file_path}; column {col_name} will be mostly NA")

climate_df = climate_df.sort_values("date").reset_index(drop=True)
climate_df.to_csv("data/raw/climate_disaster_monthly.csv", index=False)
print("saved data/raw/climate_disaster_monthly.csv", climate_df.shape)




## 5) Merge all monthly blocks into final panel


In [ ]:
panel = futures_df.merge(macro_df, on="date", how="left")
panel = panel.merge(climate_df, on="date", how="left")

required_cols = [
    "futures_return",
    "enso_oni",
    "temperature_anomaly",
    "precipitation_anomaly",
    "drought_index",
    "extreme_heat_events",
    "usd_index",
    "interest_rate",
    "vix",
]
for c in required_cols:
    if c not in panel.columns:
        panel[c] = pd.NA

panel = panel.sort_values(["commodity", "date"]).reset_index(drop=True)
panel.to_csv("data/processed/monthly_panel.csv", index=False)

print(panel.head())
print("\nMissing values by required variable:")
print(panel[required_cols].isna().sum())
print("\nSaved data/processed/monthly_panel.csv", panel.shape)
